# Кандидатогенерация (Avito, поиск услуг)

По короткому поисковому запросу отбираем до **50** `item_id` из корпуса
`benchmark_items.parquet`, которые дальше пойдут на ранжирование.
Метрика — **Recall@50** (порядок объявлений внутри топ-50 не важен).

## Подход

Итоговый скор объявления для запроса — сумма четырёх типов сигналов:

1. **Текстовое сходство**, ансамбль из двух каналов:
   - char-TF-IDF (`analyzer='char_wb'`, n-граммы 3–5 символов) — устойчив к словоформам и опечаткам;
   - semantic-канал: эмбеддинги модели `ai-forever/ru-en-RoSBERTa` — ловит смысловые совпадения (синонимы, разный порядок слов).
2. **Аддитивные бустеры** по структурированным полям (категория, локация).
3. **Географический сигнал** по медианному центру кликов для `search_location_id` (вес 0.5, масштаб 75 км по умолчанию).
4. **Бонус-канал** по истории (`train.parquet`).

## Журнал экспериментов

Все параметры запуска и метрики валидации автоматически сохраняются в файл
`experiments_log.csv` в текущей папке — это позволяет удобно сравнивать
результаты разных запусков с разными гиперпараметрами.

> Ноутбук — прямой перенос `solution.py`. Единственное содержательное отличие:
> параметры запуска заданы переменными со значениями по умолчанию в ячейке


## 1. Импорты

In [16]:
import gc
import os
import re
import csv
import time
from collections import defaultdict, Counter
from datetime import datetime

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer


## 2. Константы по умолчанию

Значения по умолчанию, используемые при запуске пайплайна.

In [ ]:
TITLE_REPEAT = 2
INFM_PARAMS_TRUNC = 500
SEM_INFM_TRUNC = 500
NGRAM_RANGE = (3, 5)
MIN_DF = 2
MAX_DF = 0.9
TOP_K = 50
CAT_BOOST = 1.0
LOC_BOOST = 0.5
GEO_WEIGHT = 0.5
GEO_SCALE_KM = 75.0
TRAIN_BONUS = 0
SIM_TFIDF_WEIGHT = 0
SIM_SEM_WEIGHT = 1.0
SEMANTIC_MODEL_NAME = "ai-forever/ru-en-RoSBERTa"

SEMANTIC_QUERY_PREFIX = "search_query: "
SEMANTIC_DOC_PREFIX = "search_document: "
SEMANTIC_BATCH_SIZE = 64
SIM_BATCH_SIZE = 150
RANDOM_SEED = 42
VALIDATE_SAMPLE_SIZE = 3000   # число QUERY-ГРУПП в валидации (не строк train)


## 3. Конфигурация запуска

Параметры запуска заданы напрямую переменными со значениями по умолчанию;
при необходимости измените их здесь перед запуском пайплайна.

In [18]:
TRAIN_PATH = "train.parquet"
QUERIES_PATH = "benchmark_queries.parquet"
ITEMS_PATH = "benchmark_items.parquet"
OUTPUT_PATH = "answer.csv"
RUN_NAME = None
NO_SEMANTIC = False
use_semantic_prefix = True
SEMANTIC_DESC_CHARS = 300
VALIDATE = True

## 4. Логирование в CSV

In [19]:
def log_experiment_to_csv(metrics, total_time):
    """Сохраняет гиперпараметры и метрики эксперимента в CSV-файл."""
    filename = "experiments_log.csv"

    # Базовые поля
    fieldnames = [
        "timestamp", "run_name",
        "sim_tfidf_weight", "sim_sem_weight", "cat_boost", "loc_boost", "train_bonus",
        "geo_weight", "geo_scale_km",
        "ngram_min", "ngram_max", "min_df", "max_df",
        "use_semantic", "semantic_model", "semantic_use_prefix", "semantic_desc_chars",
        "total_runtime_sec"
    ]
    # Поля метрик валидации (добавляются, если были вычислены)
    if metrics:
        fieldnames.extend([
            "val_protocol",
            "val_recall_no_bonus", "val_recall_with_bonus", "val_hit_rate_with_bonus",
            "val_bonus_coverage", "val_bench_bonus_coverage", "val_recall_est_bench_coverage",
            "val_queries", "val_corpus_size",
            "val_recall_has_bonus_true",
            "val_recall_has_bonus_true_no_bonus_signal", "val_recall_has_bonus_false"
        ])

    file_exists = os.path.isfile(filename)

    if file_exists:
        with open(filename, mode='r', newline='', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            old_fieldnames = [name.strip() for name in (reader.fieldnames or [])]
            old_rows = [
                {key.strip(): value for key, value in row.items() if key is not None}
                for row in reader
            ]
        merged_fieldnames = list(dict.fromkeys(old_fieldnames + fieldnames))
        if merged_fieldnames != old_fieldnames:
            with open(filename, mode='w', newline='', encoding='utf-8') as f:
                writer = csv.DictWriter(f, fieldnames=merged_fieldnames)
                writer.writeheader()
                writer.writerows(old_rows)
        fieldnames = merged_fieldnames

    with open(filename, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()

        # Формируем строку данных
        row = {
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "run_name": RUN_NAME or "unnamed_run",
            "sim_tfidf_weight": SIM_TFIDF_WEIGHT,
            "sim_sem_weight": SIM_SEM_WEIGHT,
            "cat_boost": CAT_BOOST,
            "loc_boost": LOC_BOOST,
            "geo_weight": GEO_WEIGHT,
            "geo_scale_km": GEO_SCALE_KM,
            "train_bonus": TRAIN_BONUS,
            "ngram_min": NGRAM_RANGE[0],
            "ngram_max": NGRAM_RANGE[1],
            "min_df": MIN_DF,
            "max_df": MAX_DF,
            "use_semantic": not NO_SEMANTIC,
            "semantic_model": SEMANTIC_MODEL_NAME,
            "semantic_use_prefix": use_semantic_prefix,
            "semantic_desc_chars": SEMANTIC_DESC_CHARS,
            "total_runtime_sec": round(total_time, 2)
        }

        # Добавляем метрики, округляя float для читаемости
        if metrics:
            for k, v in metrics.items():
                row[k] = round(v, 4) if isinstance(v, float) else v

        writer.writerow(row)

    print(f"\n📊 Результаты эксперимента сохранены в: {os.path.abspath(filename)}")


## 5. Текстовые утилиты

In [20]:
def normalize(s):
    if s is None:
        return ""
    s = str(s).lower().replace("ё", "е")
    s = re.sub(r"[^0-9a-zа-я\s]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def clean_light(s):
    if s is None:
        return ""
    s = str(s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def build_item_text_tfidf(title, infm_params):
    return (normalize(title) + " ") * TITLE_REPEAT + normalize(infm_params)[:INFM_PARAMS_TRUNC]

def build_query_text_tfidf(query, infm_params):
    return (normalize(query) + " ") * TITLE_REPEAT + normalize(infm_params)

def build_item_text_semantic(title, infm_params, description=None, desc_chars=0, use_prefix=True):
    t = clean_light(title)
    p = clean_light(infm_params)[:SEM_INFM_TRUNC]
    text = (t + ". " + p).strip(". ")
    if desc_chars > 0 and description:
        text = (text + ". " + clean_light(description)[:desc_chars]).strip(". ")
    if use_prefix:
        text = SEMANTIC_DOC_PREFIX + text
    return text

def build_query_text_semantic(query, infm_params, use_prefix=True):
    q = clean_light(query)
    p = clean_light(infm_params)
    text = (q + ". " + p).strip(". ")
    if use_prefix:
        text = SEMANTIC_QUERY_PREFIX + text
    return text


## 6. Semantic-канал

In [21]:
def try_load_semantic_model(model_name, disabled):
    if disabled:
        print("Semantic-канал выключен флагом no_semantic.")
        return None
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError:
        print("WARNING: пакет sentence-transformers не установлен. Работаем на чистом TF-IDF.")
        return None
    try:
        model = SentenceTransformer(model_name)
        return model
    except Exception as e:
        print(f"WARNING: не удалось загрузить модель '{model_name}' ({e}). Работаем на чистом TF-IDF.")
        return None

def encode_semantic(model, texts, batch_size=SEMANTIC_BATCH_SIZE):
    emb = model.encode(
        texts, batch_size=batch_size, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    )
    return emb.astype(np.float32)

def zscore_rows(mat):
    mean = mat.mean(axis=1, keepdims=True)
    std = mat.std(axis=1, keepdims=True)
    std[std < 1e-9] = 1.0
    return (mat - mean) / std


## 7. Бонус-канал по истории (`train.parquet`)

In [22]:
def load_train_bonus_source(train_path):
    cols = ["search_query", "item_id", "search_infm_params_text",
            "search_category", "search_location_id",
            "item_latitude", "item_longitude"]
    train_df = pd.read_parquet(train_path, columns=cols, engine="pyarrow")
    for col in ["item_latitude", "item_longitude"]:
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
    return train_df

def build_q2counts(train_df):
    q2counts = defaultdict(Counter)
    for q, i in zip(train_df["search_query"].tolist(), train_df["item_id"].tolist()):
        q2counts[q][i] += 1
    return q2counts

def bonus_items_for_query(q2counts, query_text, item_id_to_idx, exclude_item=None):
    counts = q2counts.get(query_text)
    if not counts:
        return []
    if exclude_item is not None and counts.get(exclude_item, 0) > 0:
        counts = counts.copy()
        counts[exclude_item] -= 1
        if counts[exclude_item] <= 0:
            del counts[exclude_item]
    return [item_id_to_idx[iid] for iid in counts.keys() if iid in item_id_to_idx]


## 8. Ядро скоринга

In [23]:
def compute_base_sim_batch(X_q_tfidf, q_sem_emb, item_cat, item_loc, item_emb,
                            X_items_T, cats, locs, geo_centers,
                            item_lat, item_lon, use_semantic):
    sim_tfidf = (X_q_tfidf @ X_items_T).toarray()
    sim = zscore_rows(sim_tfidf) * SIM_TFIDF_WEIGHT

    if use_semantic:
        sim_sem = q_sem_emb @ item_emb.T
        sim += zscore_rows(sim_sem) * SIM_SEM_WEIGHT

    sim += CAT_BOOST * (item_cat[None, :] == cats[:, None])
    sim += LOC_BOOST * (item_loc[None, :] == locs[:, None])
    valid_item_geo = np.isfinite(item_lat) & np.isfinite(item_lon)
    for row_idx, (center_lat, center_lon) in enumerate(geo_centers):
        if not np.isfinite(center_lat) or not np.isfinite(center_lon):
            continue
        dy = (item_lat - center_lat) * 111.32
        dx = (item_lon - center_lon) * 111.32 * np.cos(np.deg2rad(center_lat))
        geo_score = np.exp(-np.hypot(dx, dy) / GEO_SCALE_KM)
        geo_score[~valid_item_geo] = 0.0
        sim[row_idx] += GEO_WEIGHT * geo_score
    return sim

def geo_centers_for_locations(locations, geo_centers):
    return geo_centers.reindex(locations)[["item_latitude", "item_longitude"]].to_numpy(dtype=np.float32)

def top_k_row(row, k=TOP_K):
    top_idx = np.argpartition(-row, k)[:k]
    return top_idx[np.argsort(-row[top_idx])]


## 9. Offline-валидация (hold-out по группам запросов)

* Train делится **по группам запросов** (ключ = все `search_*` признаки). Валидационные группы
  целиком исключаются из бонус-канала и из геоцентров — они строятся только по остальному train.
* Корпус оценки = `benchmark_items` + все релевантные объявления валидационных запросов
  (размер ≈ корпуса бенчмарка), поэтому нужное объявление всегда физически есть в корпусе.
* Recall@50 — по запросам: `|top50 ∩ relevant| / |relevant|`, усреднение по запросам.
* Доля запросов с активным бонус-каналом сравнивается с реальной долей в `benchmark_queries`;
  итоговая оценка `val_recall_est_bench_coverage` пересчитывается на реальную долю.


In [24]:
QUERY_KEY_COLS = ["search_query", "search_location_id", "search_is_delivery_search",
                  "search_infm_params_text", "search_category"]
VAL_TRAIN_COLS = QUERY_KEY_COLS + [
    "item_id", "item_title_raw", "item_description_raw", "item_infm_params_text",
    "item_category_id", "item_location_id", "item_latitude", "item_longitude",
]

def load_train_for_validation(train_path):
    """Train со всеми полями запроса и объявления (нужны тексты объявлений для eval-корпуса)."""
    df = pd.read_parquet(train_path, columns=VAL_TRAIN_COLS, engine="pyarrow")
    for col in ["item_latitude", "item_longitude"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

def make_query_key(df):
    """Ключ группы запроса = все search_* признаки. Строки с одним ключом - один запрос
    с несколькими выбранными объявлениями (аналог query_id в бенчмарке)."""
    key = df[QUERY_KEY_COLS[0]].astype(str)
    for c in QUERY_KEY_COLS[1:]:
        key = key + "\x1f" + df[c].astype(str)
    return key.to_numpy()

def split_query_groups(keys, n_val_groups, rng):
    """Возвращает булеву маску строк, принадлежащих валидационным группам.
    Группа целиком попадает либо в val, либо в fit - строки одного запроса не делятся."""
    codes, uniques = pd.factorize(keys)
    n_groups = len(uniques)
    n_val = int(min(n_val_groups, max(1, n_groups // 10)))   # fit остаётся >= 90% групп
    val_groups = rng.choice(n_groups, size=n_val, replace=False)
    return np.isin(codes, val_groups)

def run_offline_validation(train_path, item_ids, item_cat, item_loc, item_lat, item_lon,
                            item_emb, Xt, vec, sem_model, use_semantic, use_semantic_prefix,
                            n_val_queries, bench_bonus_coverage, rng):
    print("\n=== Offline-валидация: hold-out по группам запросов ===")
    train_v = load_train_for_validation(train_path)
    keys = make_query_key(train_v)
    is_val = split_query_groups(keys, n_val_queries, rng)
    fit_df, val_df = train_v[~is_val], train_v[is_val].copy()
    val_df["_qkey"] = keys[is_val]


    # бонус-канал и геоцентры - ТОЛЬКО по fit-части (валидационные запросы для них не существуют)
    q2counts_fit = build_q2counts(fit_df)
    geo_fit = fit_df.dropna(subset=["search_location_id"])
    geo_fit = geo_fit[np.isfinite(geo_fit["item_latitude"]) & np.isfinite(geo_fit["item_longitude"])]
    geo_centers_fit = geo_fit.groupby("search_location_id")[["item_latitude", "item_longitude"]].median()

    # --- корпус оценки: benchmark_items + релевантные объявления val-запросов ---
    bench_id_set = set(item_ids)
    extra = val_df.drop_duplicates("item_id")
    extra = extra[~extra["item_id"].isin(bench_id_set)]
    n_extra = len(extra)
    if n_extra > 0:
        ex_tfidf = [build_item_text_tfidf(t, p) for t, p in
                    zip(extra["item_title_raw"].tolist(), extra["item_infm_params_text"].tolist())]
        ev_Xt = sparse.hstack([Xt, vec.transform(ex_tfidf).T.tocsr()], format="csr")
        ev_ids = np.concatenate([item_ids, extra["item_id"].to_numpy()])
        ev_cat = np.concatenate([item_cat, extra["item_category_id"].to_numpy()])
        ev_loc = np.concatenate([item_loc, extra["item_location_id"].to_numpy()])
        ev_lat = np.concatenate([item_lat, extra["item_latitude"].to_numpy(dtype=np.float32)])
        ev_lon = np.concatenate([item_lon, extra["item_longitude"].to_numpy(dtype=np.float32)])
        if use_semantic:
            descs = extra["item_description_raw"].tolist() if SEMANTIC_DESC_CHARS > 0 else [None] * n_extra
            ex_sem = [build_item_text_semantic(t, p, d, SEMANTIC_DESC_CHARS, use_semantic_prefix)
                      for t, p, d in zip(extra["item_title_raw"].tolist(),
                                         extra["item_infm_params_text"].tolist(), descs)]
            ev_emb = np.vstack([item_emb, encode_semantic(sem_model, ex_sem)])
        else:
            ev_emb = None
    else:
        ev_Xt, ev_ids, ev_cat, ev_loc, ev_lat, ev_lon, ev_emb = Xt, item_ids, item_cat, item_loc, item_lat, item_lon, item_emb
    ev_id2idx = {iid: i for i, iid in enumerate(ev_ids)}
    print(f"  групп запросов: fit={len(set(keys[~is_val]))}, val={len(set(keys[is_val]))}")
    print(f"  корпус оценки: {len(ev_ids)} объявлений (benchmark {len(item_ids)} + релевантные val вне benchmark {n_extra})")

    # --- уникальные val-запросы и полные множества релевантных объявлений ---
    val_q = val_df.drop_duplicates("_qkey").reset_index(drop=True)
    rel_lists = val_df.groupby("_qkey", sort=False)["item_id"].agg(lambda x: list(dict.fromkeys(x)))
    val_q["_rel"] = val_q["_qkey"].map(rel_lists)
    rel_idx = [np.array([ev_id2idx[i] for i in rel], dtype=np.int64) for rel in val_q["_rel"]]

    p_query = val_q["search_query"].tolist()
    p_infm = val_q["search_infm_params_text"].tolist()
    p_cat = val_q["search_category"].to_numpy()
    p_loc = val_q["search_location_id"].to_numpy()
    X_p = vec.transform([build_query_text_tfidf(q, p) for q, p in zip(p_query, p_infm)])
    p_emb = encode_semantic(sem_model, [build_query_text_semantic(q, p, use_prefix=use_semantic_prefix)
                                         for q, p in zip(p_query, p_infm)]) if use_semantic else None

    rec_nb, rec_wb, hit_wb, has_bonus = [], [], [], []
    n = len(val_q)
    for start in range(0, n, SIM_BATCH_SIZE):
        end = min(start + SIM_BATCH_SIZE, n)
        geo_batch = geo_centers_for_locations(p_loc[start:end], geo_centers_fit)
        sim = compute_base_sim_batch(
            X_p[start:end], p_emb[start:end] if use_semantic else None, ev_cat, ev_loc, ev_emb,
            ev_Xt, p_cat[start:end], p_loc[start:end], geo_batch, ev_lat, ev_lon, use_semantic,
        )
        for i in range(end - start):
            qi, rel = start + i, rel_idx[start + i]
            top_nb = top_k_row(sim[i])
            bonus_idxs = bonus_items_for_query(q2counts_fit, p_query[qi], ev_id2idx)  # без exclude: val уже вне fit
            if bonus_idxs:
                row = sim[i].copy()
                row[bonus_idxs] += TRAIN_BONUS
                top_wb = top_k_row(row)
            else:
                top_wb = top_nb
            # Recall@50 запроса = |top50 ∩ relevant| / |relevant|
            rec_nb.append(np.isin(rel, top_nb).mean())
            r = np.isin(rel, top_wb).mean()
            rec_wb.append(r); hit_wb.append(r > 0); has_bonus.append(len(bonus_idxs) > 0)

    rec_nb, rec_wb, hit_wb, has_bonus = map(np.array, (rec_nb, rec_wb, hit_wb, has_bonus))
    cov = float(has_bonus.mean())
    if has_bonus.any() and (~has_bonus).any():   # пересчёт на реальную долю «тёплых» запросов бенчмарка
        est = bench_bonus_coverage * rec_wb[has_bonus].mean() + (1 - bench_bonus_coverage) * rec_wb[~has_bonus].mean()
    else:
        est = rec_wb.mean()

    metrics = {
        "val_protocol": "query_group_holdout_v2",
        "val_recall_no_bonus": float(rec_nb.mean()),
        "val_recall_with_bonus": float(rec_wb.mean()),
        "val_hit_rate_with_bonus": float(hit_wb.mean()),
        "val_bonus_coverage": cov,
        "val_bench_bonus_coverage": float(bench_bonus_coverage),
        "val_recall_est_bench_coverage": float(est),
        "val_queries": float(n),
        "val_corpus_size": float(len(ev_ids)),
    }
    if has_bonus.any():
        metrics["val_recall_has_bonus_true"] = float(rec_wb[has_bonus].mean())
        metrics["val_recall_has_bonus_true_no_bonus_signal"] = float(rec_nb[has_bonus].mean())
    if (~has_bonus).any():
        metrics["val_recall_has_bonus_false"] = float(rec_wb[~has_bonus].mean())

    print(f"\n  Запросов: {n}, релевантных на запрос в среднем: {np.mean([len(r) for r in rel_idx]):.2f}")
    print(f"  Recall@50 без бонус-канала:                {metrics['val_recall_no_bonus']:.4f}")
    print(f"  Recall@50 с бонус-каналом (fit-only):      {metrics['val_recall_with_bonus']:.4f}")
    print(f"  HitRate@50 (хотя бы одно релевантное):     {metrics['val_hit_rate_with_bonus']:.4f}")
    print(f"  Активность бонус-канала: val={cov:.1%}, реальный бенчмарк={bench_bonus_coverage:.1%}")
    if "val_recall_has_bonus_true" in metrics:
        print(f"    - has_bonus=True:  {metrics['val_recall_has_bonus_true']:.4f} "
              f"(без бонуса на этих же запросах: {metrics['val_recall_has_bonus_true_no_bonus_signal']:.4f})")
    if "val_recall_has_bonus_false" in metrics:
        print(f"    - has_bonus=False: {metrics['val_recall_has_bonus_false']:.4f}")
    print(f"  >>> Оценка Recall@50 при реальной доле бонуса бенчмарка: {est:.4f}")

    del train_v, fit_df, val_df, ev_Xt, ev_emb
    gc.collect()
    return metrics


## 10. Запуск пайплайна

Дальше — тело бывшей функции `main()`, разбитое на те же 7 шагов, что были
отмечены комментариями в скрипте. Ячейки выполняются последовательно и
используют значения из раздела «Конфигурация запуска».

In [25]:
rng = np.random.RandomState(RANDOM_SEED)
t0 = time.time()
val_metrics = {}


### Шаг 1. Загрузка объявлений (корпус)

In [26]:
print("Загрузка объявлений...")
items_cols = ["item_id", "item_title_raw", "item_infm_params_text",
              "item_category_id", "item_location_id",
              "item_latitude", "item_longitude"]
if SEMANTIC_DESC_CHARS > 0:
    items_cols.append("item_description_raw")
items = pd.read_parquet(ITEMS_PATH, columns=items_cols, engine="pyarrow")
item_ids = items["item_id"].to_numpy()
item_cat = items["item_category_id"].to_numpy()
item_loc = items["item_location_id"].to_numpy()
item_lat = pd.to_numeric(items["item_latitude"], errors="coerce").to_numpy(dtype=np.float32)
item_lon = pd.to_numeric(items["item_longitude"], errors="coerce").to_numpy(dtype=np.float32)

item_texts_tfidf = [
    build_item_text_tfidf(t, p)
    for t, p in zip(items["item_title_raw"].tolist(), items["item_infm_params_text"].tolist())
]
descs = items["item_description_raw"].tolist() if SEMANTIC_DESC_CHARS > 0 else [None] * len(items)
item_texts_sem = [
    build_item_text_semantic(t, p, d, SEMANTIC_DESC_CHARS, use_semantic_prefix)
    for t, p, d in zip(items["item_title_raw"].tolist(), items["item_infm_params_text"].tolist(), descs)
]
del items, descs
gc.collect()
print(f"  объявлений: {len(item_ids)}, время: {time.time()-t0:.1f}s")


Загрузка объявлений...
  объявлений: 189212, время: 25.6s


### Шаг 2. Загрузка запросов бенчмарка

In [27]:
print("Загрузка запросов...")
queries = pd.read_parquet(QUERIES_PATH, engine="pyarrow")
query_ids = queries["query_id"].to_numpy()
query_cat = queries["search_category"].to_numpy()
query_loc = queries["search_location_id"].to_numpy()

query_texts_tfidf = [
    build_query_text_tfidf(q, p)
    for q, p in zip(queries["search_query"].tolist(), queries["search_infm_params_text"].tolist())
]
query_texts_sem = [
    build_query_text_semantic(q, p, use_prefix=use_semantic_prefix)
    for q, p in zip(queries["search_query"].tolist(), queries["search_infm_params_text"].tolist())
]
query_texts_raw = queries["search_query"].tolist()
del queries
gc.collect()
print(f"  запросов: {len(query_ids)}, время: {time.time()-t0:.1f}s")


Загрузка запросов...
  запросов: 2452, время: 25.9s


### Шаг 3. TF-IDF: словарь по items + queries (опционально)

In [28]:
print("Построение TF-IDF (словарь: объявления + запросы)...")
vec = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(NGRAM_RANGE[0], NGRAM_RANGE[1]),
    min_df=MIN_DF, max_df=MAX_DF, sublinear_tf=True,
)
vec.fit(item_texts_tfidf + query_texts_tfidf)
X_items = vec.transform(item_texts_tfidf)
X_queries = vec.transform(query_texts_tfidf)
del item_texts_tfidf, query_texts_tfidf
gc.collect()
Xt = X_items.T.tocsr()
print(f"  матрица объявлений: {X_items.shape}, словарь: {len(vec.vocabulary_)}, время: {time.time()-t0:.1f}s")


Построение TF-IDF (словарь: объявления + запросы)...
  матрица объявлений: (189212, 169788), словарь: 169788, время: 144.1s


### Шаг 4. Semantic-канал

In [ ]:
sem_model = try_load_semantic_model(SEMANTIC_MODEL_NAME, NO_SEMANTIC)
use_semantic = sem_model is not None
item_emb = query_emb = None
if use_semantic:
    print(f"Semantic-канал: кодируем объявления моделью '{SEMANTIC_MODEL_NAME}'...")
    item_emb = encode_semantic(sem_model, item_texts_sem, SEMANTIC_BATCH_SIZE)
    print(f"  готово, время: {time.time()-t0:.1f}s")
    print("Semantic-канал: кодируем запросы бенчмарка...")
    query_emb = encode_semantic(sem_model, query_texts_sem, SEMANTIC_BATCH_SIZE)
    print(f"  готово, время: {time.time()-t0:.1f}s")
del item_texts_sem, query_texts_sem
gc.collect()


Some weights of RobertaModel were not initialized from the model checkpoint at ai-forever/ru-en-RoSBERTa and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Semantic-канал: кодируем объявления моделью 'ai-forever/ru-en-RoSBERTa'...


Batches:   0%|          | 0/2957 [00:00<?, ?it/s]

### Шаг 5. Бонус-канал из train + offline-валидация (опционально)

In [ ]:
print("Загрузка train.parquet (для бонус-канала)...")
train_df = load_train_bonus_source(TRAIN_PATH)
q2counts = build_q2counts(train_df)
geo_train = train_df.dropna(subset=["search_location_id"])
geo_train = geo_train[np.isfinite(geo_train["item_latitude"]) & np.isfinite(geo_train["item_longitude"])]
geo_centers = geo_train.groupby("search_location_id")[["item_latitude", "item_longitude"]].median()
print(f"  географических центров по search_location_id: {len(geo_centers)}")
item_id_to_idx = {iid: idx for idx, iid in enumerate(item_ids)}
print(f"  уникальных текстов запросов в train: {len(q2counts)}, время: {time.time()-t0:.1f}s")

if VALIDATE:
    bench_bonus_coverage = float(np.mean([
        len(bonus_items_for_query(q2counts, t, item_id_to_idx)) > 0 for t in query_texts_raw
    ]))
    print(f"  бонус-канал активен на {bench_bonus_coverage:.1%} запросов бенчмарка")
    val_metrics = run_offline_validation(
        TRAIN_PATH, item_ids, item_cat, item_loc, item_lat, item_lon,
        item_emb, Xt, vec, sem_model, use_semantic, use_semantic_prefix,
        VALIDATE_SAMPLE_SIZE, bench_bonus_coverage, rng,
    )
del train_df
gc.collect()


Загрузка train.parquet (для бонус-канала)...
  географических центров по search_location_id: 2782
  уникальных текстов запросов в train: 74529, время: 644.4s
  бонус-канал активен на 14.7% запросов бенчмарка

=== Offline-валидация: hold-out по группам запросов ===


Batches:   0%|          | 0/57 [00:00<?, ?it/s]

  групп запросов: fit=351463, val=3000
  корпус оценки: 192819 объявлений (benchmark 189212 + релевантные val вне benchmark 3607)


Batches:   0%|          | 0/47 [00:00<?, ?it/s]


  Запросов: 3000, релевантных на запрос в среднем: 1.29
  Recall@50 без бонус-канала:                0.9001
  Recall@50 с бонус-каналом (fit-only):      0.9001
  HitRate@50 (хотя бы одно релевантное):     0.9100
  Активность бонус-канала: val=62.9%, реальный бенчмарк=14.7%
    - has_bonus=True:  0.9035 (без бонуса на этих же запросах: 0.9035)
    - has_bonus=False: 0.8944
  >>> Оценка Recall@50 при реальной доле бонуса бенчмарка: 0.8957
  ВНИМАНИЕ: доля активного бонуса в val сильно отличается от бенчмарка - ориентируйтесь на строку с >>>.
  Валидационные запросы исключены из бонус-канала и геоцентров; корпус содержит все их релевантные объявления.



0

### Шаг 6. Расчёт top-50 для каждого запроса

In [ ]:
print("Расчёт сходства и отбор топ-50...")
n_queries = X_queries.shape[0]
results = [None] * n_queries
for start in range(0, n_queries, SIM_BATCH_SIZE):
    end = min(start + SIM_BATCH_SIZE, n_queries)
    cats, locs = query_cat[start:end], query_loc[start:end]
    geo_batch = geo_centers_for_locations(locs, geo_centers)
    sem_batch = query_emb[start:end] if use_semantic else None
    sim = compute_base_sim_batch(
        X_queries[start:end], sem_batch, item_cat, item_loc, item_emb,
        Xt, cats, locs, geo_batch, item_lat, item_lon, use_semantic,
    )
    for i in range(end - start):
        qi = start + i
        row = sim[i]
        bonus_idxs = bonus_items_for_query(q2counts, query_texts_raw[qi], item_id_to_idx)
        if bonus_idxs:
            row = row.copy()
            row[bonus_idxs] += TRAIN_BONUS
        results[qi] = item_ids[top_k_row(row)]
    if start % 1500 == 0:
        print(f"  обработано {start}/{n_queries}, время: {time.time()-t0:.1f}s")


Расчёт сходства и отбор топ-50...
  обработано 0/2452, время: 715.0s
  обработано 1500/2452, время: 733.2s


### Шаг 7. Сохранение `answer.csv` и лога эксперимента

In [ ]:
answer = pd.DataFrame({
    "query_id": query_ids,
    "answer": [" ".join(x) for x in results],
})
answer.to_csv(OUTPUT_PATH, index=False)
total_time = time.time() - t0
print(f"\n✅ Готово! Файл сохранён: {OUTPUT_PATH}")
print(f"   Строк: {len(answer)}, Общее время: {total_time:.1f}s")

# Логируем эксперимент в CSV
log_experiment_to_csv(val_metrics if VALIDATE else None, total_time)

answer.head()



✅ Готово! Файл сохранён: answer.csv
   Строк: 2452, Общее время: 743.0s

📊 Результаты эксперимента сохранены в: d:\VSCode\Стажа\Avito\experiments_log.csv


,query_id,answer
0,70DfDUpwjxB4lzFd,355392014208b7bf fa01652ca5f57d0f 4494aa11cbdc...
1,JTrdTaZJvSiLPkXj,422d3ffdd5bbf626 367af128a9ea2a48 cbeccbecb1fb...
2,LZCZNoVG4AFUkVRJ,dab52187b4500d9b 168a9207e80b0be4 d62074dd39ca...
3,660ac9QVtXkRxZC3,af91ec4a29b66636 bd2cb4500fad728e 3cce43c81145...
4,YgHcM9MVbxKnxD1e,13f58fe9542bdb1a ba78ad593ec3412d 615c73ea4c3b...
